# Activity 2: Web Scraping

1. **An API**: the provider sends clean data on request (FRED).
2. **A web page**: the numbers are in an HTML table (Central Bank of Trinidad and Tobago).
3. **An Excel file**: the numbers are in a workbook made for people to read, not for programs
   (CBTT Economic Datapack).

Cells marked `<YOUR CODE HERE: ...>` are for you to complete. The solution is in
`sols/act2_scraping.ipynb`.

In [ ]:
import requests
import pandas as pd
from bs4 import BeautifulSoup

## Part 1. FRED: an API

FRED is the St. Louis Fed's database. You ask for **one series by its code** and get back
dates and values as JSON. The request is a URL with three parameters:

| parameter | value |
|---|---|
| `series_id` | the series code, found on the series page at fred.stlouisfed.org |
| `api_key` | a free personal key: https://fredaccount.stlouisfed.org/apikeys |
| `file_type` | `json` |

Everything needed is in the next cell: the key, the URL, the request and the conversion to a DataFrame.

In [ ]:
API_KEY = "72304a91a13cc0ea6c610ae5845c7493"            # the instructor's key: get your own for later use
SERIES  = "DCOILBRENTEU"                                # Brent crude oil, US$ per barrel, daily

url = ("https://api.stlouisfed.org/fred/series/observations"
       f"?series_id={SERIES}&api_key={API_KEY}&file_type=json")

response = requests.get(url, timeout=60)               # send the request
print(response.status_code)                             # 200 = OK

obs = response.json()["observations"]                  # a list of {"date": ..., "value": ...}
df_fred = pd.DataFrame(obs)[["date", "value"]]
df_fred["date"]  = pd.to_datetime(df_fred["date"])
df_fred["value"] = pd.to_numeric(df_fred["value"], errors="coerce")   # FRED writes "." for missing
df_fred = df_fred.set_index("date").rename(columns={"value": SERIES})
df_fred.tail()

In [ ]:
df_fred.plot(figsize=(10, 3.5), title="Brent crude oil, US$ per barrel")

Two series in a loop and merge them. Start with an empty dataframe

In [ ]:
#<YOUR CODE HERE: download DEXUSEU and plot it>
df = pd.DataFrame()
SERIES  = ["DCOILBRENTEU", "DEXUSEU"]

for SERIE in SERIES :
    url = ("https://api.stlouisfed.org/fred/series/observations"
           f"?series_id={SERIE}&api_key={API_KEY}&file_type=json")

    response = requests.get(url, timeout=60)               # send the request

    obs = response.json()["observations"]                  # a list of {"date": ..., "value": ...}
    df_fred = pd.DataFrame(obs)[["date", "value"]]
    df_fred["date"]  = pd.to_datetime(df_fred["date"])
    df_fred["value"] = pd.to_numeric(df_fred["value"], errors="coerce")   # FRED writes "." for missing
    df_fred = df_fred.set_index("date").rename(columns={"value": SERIE})
    df = pd.concat([df, df_fred], axis=1)

df.plot()


## Part 2. A web page: the CBTT Data Centre

The Central Bank publishes its statistics as web pages, one table per page. We take the
**consumer price index** (monthly).

### Step 1: look before you code

Open the page in your browser:

https://www.central-bank.org.tt/data-center-block/?title=consumer-price-index-index-values-monthly&id=36

Right-click the table and choose **Inspect**. The table is a `<table id="table_1">`, the column
names are `<th>` cells inside `<thead>`, and each row of data is a `<tr>` inside `<tbody>`.

### Step 2: request the page

`headers` makes the request look like it comes from a browser. Many sites block requests without it.

In [ ]:
url = "https://www.central-bank.org.tt/data-center-block/?title=consumer-price-index-index-values-monthly&id=36"
hdr = {"User-Agent": "Mozilla/5.0"}

r = requests.get(url, headers=hdr, timeout=60)
print(r.status_code)                       # 200 = OK, 403 = blocked, 404 = not found
print(r.headers["Content-Type"])           # text/html: a web page, not data

### Step 3: html text

How does a website code look like?

In [ ]:
print(r.text[:500])

### Step 4: find the table

`BeautifulSoup` turns the text into a tree we can search. `select_one("table#table_1")` means:
the `<table>` whose `id` is `table_1`.

In [ ]:
soup  = BeautifulSoup(r.text, "html.parser")
table = soup.select_one("table#table_1")
print(str(table)[:500])

### Step 5: the column names

Every `<th>` inside `<thead>`. `get_text(strip=True)` keeps only the text, without tags or spaces.

In [ ]:
cols = [th.get_text(strip=True) for th in table.select("thead th")]
print(len(cols), "columns")
cols[:4]

### Step 6: one row

Every `<tr>` inside `<tbody>` is a row, and every `<td>` inside it is a cell. Look at the first one.

In [ ]:
rows_html = table.select("tbody tr")
print(len(rows_html), "rows")

first = [td.get_text(strip=True) for td in rows_html[0].select("td")]
first[:4]

### Step 7: Get the rows

**COMPLETE**: repeat Step 6 for every row, then build the DataFrame.

In [ ]:
rows = []
for tr in rows_html:
    rows.append(<YOUR CODE HERE: the text of every td in tr, as in Step 6>)

df_cpi = pd.DataFrame(rows, columns=cols)
df_cpi.head()

### Step 8: clean

Everything arrived as **text**. Dates are written day first (`31/01/1991`), and the bank writes
`n.d.` where there is no data. `errors="coerce"` turns anything that is not a number into `NaN`.
The rows are not in date order on the page, so sort them.

In [ ]:
df_cpi["Date"] = pd.to_datetime(df_cpi["Date"], dayfirst=True)
df_cpi = df_cpi.set_index("Date").sort_index()
df_cpi = df_cpi.apply(pd.to_numeric, errors="coerce")
df_cpi.dtypes.head()

Show the result

## Part 3. An Excel file: the CBTT Economic Datapack

The Central Bank's **Economic Datapack** is one workbook with most of its statistics, linked from

https://www.central-bank.org.tt/resources-category/publications-and-research/all-reports/economic-datapack-report/

Workbooks like this are made to be read by people: titles, notes, headers over several rows,
empty columns between blocks. We clean one sheet step by step, looking at it after each step.

### Step 1: download the file

When the address is a file, `requests.get` returns the file itself. Save it to disk.

In [ ]:
xl_url = "https://www.central-bank.org.tt/wp-content/uploads/2026/10/edp-download-september-2026.xlsm"

r = requests.get(xl_url, headers=hdr, timeout=120)
print(r.status_code, f"{len(r.content):,} bytes")

with open("../raw/cbtt_datapack.xlsm", "wb") as f:
    f.write(r.content)

### Step 2: list the sheets

`pd.ExcelFile` opens the workbook without reading every sheet.

In [ ]:
xls = pd.ExcelFile("../raw/cbtt_datapack.xlsm")
xls.sheet_names

### Step 3: read one sheet as it is

`PAGE 5_DomesticAct_Quarterly` holds the quarterly GDP growth. `header=None` reads every row as
data, so we see the sheet exactly as it looks in Excel.

In [ ]:
raw = pd.read_excel(xls, sheet_name="PAGE 5_DomesticAct_Quarterly", header=None)
print(raw.shape)
raw.iloc[:14, :6]

Rows 0 to 6 are a description (unit, frequency, source, ...). Row 7, `Series Descriptions`, has
one clean name per column. Rows 8 to 10 are the header as people see it, spread over three rows.
The data start at row 11.

### Step 4: use row 7 as the column names

In [ ]:
df_xl = raw.copy()
df_xl.columns = raw.iloc[7].tolist()
df_xl.iloc[:14, :6]

### Step 5: keep only the data rows

In [ ]:
df_xl = df_xl.iloc[11:]
df_xl.head()

### Step 6: the dates as the index

The first column holds the dates. Each quarter is dated by the first day of its last month (March, June, ...).

In [ ]:
df_xl = df_xl.rename(columns={"Series Descriptions": "date"})
df_xl["date"] = pd.to_datetime(df_xl["date"])
df_xl = df_xl.set_index("date")
df_xl.head()

### Step 7: drop the empty columns and rows

The sheet has empty columns between blocks, and the early quarters and the latest ones have no data.
**COMPLETE**: `dropna(how="all")` drops a row only if every value in it is missing; `axis=1` does the same for columns.

In [ ]:
df_xl = <YOUR CODE HERE: drop the columns where every value is missing>
df_xl = <YOUR CODE HERE: drop the rows where every value is missing>
print(df_xl.shape)
df_xl.head()

### Step 8: Formatting

The columns came from mixed rows (text and numbers), so pandas stored them as `object`. Convert them.

In [ ]:
print(df_xl.dtypes.head(3))
df_xl = df_xl.apply(pd.to_numeric, errors="coerce")
df_xl.dtypes.head(3)

GDP growth here is the change against the same quarter a year earlier, in per cent.
**COMPLETE**: plot total, energy and non-energy GDP growth.

In [ ]:
<YOUR CODE HERE: plot the columns QGDP - Total, QGDP - Energy and QGDP - Non-Energy>

### Take-aways

| Source | Tool | Main step |
|---|---|---|
| API | `requests` + `.json()` | build the URL |
| Web page | `requests` + `BeautifulSoup` | find the table, read the rows |
| Excel | `pd.read_excel(header=None)` | look at the sheet, then cut it down |

**Look before you code**: open the page or the workbook first, then write the code.